# Phase 10 Lab — Reference Solution

**Phase:** Real-World ML and MLOps  
**Scenario:** A churn model must move from experiment to a safe, observable API with repeatable releases.

**Deliverable:** A package, artifact, API contract, tests, container, rollout plan, monitoring specification, and model card.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Move feature and training logic into importable modules.
2. Validate training and inference schemas.
3. Train and version a complete preprocessing pipeline.
4. Test feature order, edge cases, and serialization parity.
5. Define typed API requests/responses and health endpoints.
6. Design CI/CD, canary, rollback, and ownership.
7. Define service/data/model/decision monitoring and governance.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
from pathlib import Path
from src.production_example.config import Settings
from src.production_example.train import train
from src.production_example.features import MODEL_FEATURES
from src.course_utils import population_stability_index

settings=Settings(model_path=ARTIFACT_DIR/"phase10_churn_pipeline.joblib")
metrics=train(DATA_DIR/"customer_churn.csv",settings)
print("Training metrics:",metrics)
print("Model features:",MODEL_FEATURES)
assert settings.model_path.exists()

reference=np.random.default_rng(42).normal(70,20,3000)
current=np.random.default_rng(43).normal(78,24,1500)
print("Example monthly-charge PSI:",population_stability_index(reference,current))

release_checklist=pd.DataFrame([
    ["Data contract tests","pass"],
    ["Unit and integration tests","required"],
    ["Artifact hash and lineage","required"],
    ["Offline metric gate","required"],
    ["Canary and rollback","defined"],
    ["Monitoring and owner","defined"],
    ["Model card and approval","required"],
],columns=["gate","status"])
display(release_checklist)
print("See src/production_example, Dockerfile, tests, and .github/workflows/ci.yml.")

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.